The motivation for exploring Mixture of Experts (MoE) approaches emerged from two observations: the distribution of rushing yards identified during EDA and a quantitative diagnosis of the baseline model (MLP). The diagnostic analysis revealed that the CRPS for short plays (0–5 yards) was 0.00559, whereas for long plays (>15 yards) it reached 0.08845, which's a 15fold difference. Calibration analysis confirmed the same systematic bias that the predicted probability of gaining more than 15 yards was 0.0083, while the observed frequency was 0.0404. The model was systematically underestimating explosive plays, and this error appeared to be structural rather than a consequence of insufficient training or suboptimal hyperparameter choices.

This pattern suggested that the yardage distribution could not be adequately modeled by a single network trained with a uniform loss function. The behavior around the center of the distribution (short runs with high probability mass) and in the tails (long plays characterized by slower probability decay) is fundamentally different. Forcing a single model to capture both regimes inevitably leads to trade-offs that disproportionately harm tail performance, where data are scarce but tactical importance is high.

The first approach investigated was the Mixture Density Network (MDN), which can be viewed as an implicit form of Mixture of Experts. Rather than relying on separate expert networks, an MDN learns the parameters of K Gaussian distributions together with their mixture weights. The final CDF is computed analytically as a weighted combination of the components. The gating mechanism is learned jointly with the Gaussian parameters by the same network. This eliminates the need for sequential training and allows specialization across different regions of the distribution to emerge naturally during optimization.
We used 4 gaussians (low to avoid instability) and replaced only the final layer of the pre-trained MLP (the output head) while preserving the intermediate hidden layers, which had already learned rich feature representations from the data.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle
import math

In [2]:
def prepare_data(df_train, df_val):
    drop_cols = ['PlayId', 'GameId', 'Yards']
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    pos_cols = [c for c in feature_cols if 'Position' in c]
    le = LabelEncoder()
    all_pos_values = pd.concat([df_train[c] for c in pos_cols]).fillna('UNK')
    le.fit(all_pos_values)
    df_train = df_train.copy()
    df_val = df_val.copy()

    for col in pos_cols:
        df_train[col] = le.transform(df_train[col].fillna('UNK'))
        df_val[col] = df_val[col].fillna('UNK').apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else 0)

    df_train = df_train.fillna(0.0)
    df_val = df_val.fillna(0.0)
    X_train = df_train[feature_cols].values.astype(np.float32)
    X_val = df_val[feature_cols].values.astype(np.float32)
    y_train = df_train['Yards'].values.astype(np.float32)
    y_val = df_val['Yards'].values.astype(np.float32)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    return X_train, y_train, X_val, y_val, scaler, le, feature_cols

In [3]:
def get_y_col_indices(feature_cols):
    flip_keywords = ['_Y_rel', '_Sy', '_Ori_sin', '_Dir_sin']
    return [i for i, c in enumerate(feature_cols)
            if any(c.endswith(kw) for kw in flip_keywords)]

In [4]:
#augmentation by flipping Y axis
class PlayDataset(Dataset):
    def __init__(self, X, y, y_flip_cols=None, is_train=True):
        self.is_train = is_train
        self.flip_idx = list(y_flip_cols) if y_flip_cols else []
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]
        if self.is_train and torch.rand(1).item() > 0.5:
            x[self.flip_idx] = -x[self.flip_idx]
        return x, y

In [5]:
class MLPBody(nn.Module):
    #same MLP body we've been using
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ELU(),
            nn.Dropout(dropout),)

    def forward(self, x):
        return self.net(x)  

class MDNHead(nn.Module):
    #maps 256 dimensions to k Normals and returns mean and std of each, together with an associated weight
    def __init__(self, hidden_dim=256, K=4):
        super().__init__()
        self.K = K
        self.net = nn.Linear(hidden_dim, 3 * K)

    def forward(self, h):
        out = self.net(h)                          #[B, 3K]
        pi_logits = out[:, :self.K]                #[B, K] weights logits
        mu = out[:, self.K:2*self.K]        #[B, K] means
        log_sigma = out[:, 2*self.K:]              #[B, K] std logs
        pi = F.softmax(pi_logits, dim=-1)       #[B, K] sum 1
        sigma = torch.exp(log_sigma).clamp(min=0.1, max=50.0)  

        return pi, mu, sigma

In [6]:
class MLPWithMDN(nn.Module):
    #pretrained body + new class head
    def __init__(self, input_dim, K=4, dropout=0.2):
        super().__init__()
        self.body = MLPBody(input_dim, dropout)
        self.head = MDNHead(hidden_dim=256, K=K)

    def forward(self, x):
        h = self.body(x)
        return self.head(h)  #pi, mu, sigma


In [7]:
def gaussian_cdf(y, mu, sigma):
    return 0.5 * (1.0 + torch.erf((y - mu) / (sigma * math.sqrt(2))))

In [8]:
def mixture_cdf(pi, mu, sigma, y_bins):
    #CRPS for each bin
    B, K = pi.shape
    N = len(y_bins)
    pi_e = pi.unsqueeze(2)                      #[B, K, 1]
    mu_e = mu.unsqueeze(2)                      #[B, K, 1]
    sigma_e = sigma.unsqueeze(2)                   #[B, K, 1]
    y_e = y_bins.view(1, 1, N)                 #[1, 1, N]
    component_cdf = gaussian_cdf(y_e, mu_e, sigma_e)
    #weighted mixture
    cdf = (pi_e * component_cdf).sum(dim=1)

    return torch.clamp(cdf, 0.0, 1.0)

In [9]:
#loss
class MDNCRPSLoss(nn.Module):
    def __init__(self, min_yard=-99, max_yard=99, clip_min=-30, clip_max=50):
        super().__init__()
        self.clip_min = clip_min
        self.clip_max = clip_max
        y_bins = torch.arange(min_yard, max_yard + 1, dtype=torch.float32)
        self.register_buffer("y_bins", y_bins)

    def forward(self, pi, mu, sigma, y_true):
        if self.training:
            y_true = torch.clamp(y_true, self.clip_min, self.clip_max)
        cdf_pred = mixture_cdf(pi, mu, sigma, self.y_bins)        
        cdf_true = (self.y_bins[None, :] >= y_true[:, None]).float() 

        return torch.mean((cdf_pred - cdf_true) ** 2)

In [10]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pi, mu, sigma = model(x_batch)
        loss = criterion(pi, mu, sigma, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

In [11]:
def evaluate_crps(model, loader, device):
    model.eval()
    y_bins_np = np.arange(-99, 100, dtype=np.float32)
    y_bins_t  = torch.tensor(y_bins_np).to(device)
    all_cdfs, all_y = [], []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            pi, mu, sigma = model(x_batch)
            cdfs = mixture_cdf(pi, mu, sigma, y_bins_t).cpu().numpy()
            all_cdfs.append(cdfs)
            all_y.append(y_batch.numpy())

    cdfs = np.vstack(all_cdfs)
    y_true = np.concatenate(all_y)
    cdf_true = (y_bins_np[None, :] >= y_true[:, None]).astype(np.float32)
    return float(np.mean((cdfs - cdf_true) ** 2)), cdfs, y_true

In [12]:
def print_tail_diagnostics(cdfs, y_val, label=""):
    y_bins_np = np.arange(-99, 100, dtype=np.float32)

    def crps_mask(cdfs, y_true, mask):
        if mask.sum() == 0:
            return float('nan')

        true_cdf = (y_bins_np[None, :] >= y_true[mask, None]).astype(np.float32)
        return float(np.mean((cdfs[mask] - true_cdf) ** 2))

    negative_mask = y_val < 0
    short_mask = (y_val >= 0) & (y_val <= 5)
    medium_mask = (y_val > 5)  & (y_val <= 15)
    long_mask = y_val > 15

    print(f"\nCRPS by region ({label})")
    print(f"Negative plays (< 0):  "
        f"n={negative_mask.sum():4d} | "
        f"CRPS={crps_mask(cdfs, y_val, negative_mask):.5f}")
    print(f"Short plays (0 - 5):   "
        f"n={short_mask.sum():4d} | "
        f"CRPS={crps_mask(cdfs, y_val, short_mask):.5f}")
    print(f"Medium plays (6 - 15): "
        f"n={medium_mask.sum():4d} | "
        f"CRPS={crps_mask(cdfs, y_val, medium_mask):.5f}")
    print(f"Long plays (> 15):     "
        f"n={long_mask.sum():4d} | "
        f"CRPS={crps_mask(cdfs, y_val, long_mask):.5f}")
    print(f"\nRight-tail calibration")
    print(f"  {'Threshold':>10} | {'Predicted':>8} | {'Observed':>8} | {'Error':>8}")

    for threshold in [10, 15, 20, 25, 30, 40]:
        idx = threshold + 99
        predicted_prob = float(np.mean(1 - cdfs[:, idx]))
        observed_prob = float(np.mean(y_val > threshold))
        error = predicted_prob - observed_prob

        print(f"  yards > {threshold:2d}   | "
            f"{predicted_prob:8.4f} | "
            f"{observed_prob:8.4f} | "
            f"{error:+8.4f}")

In [13]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Rodando no dispositivo: {device}")

df_train = pd.read_parquet("/Users/llbm/Desktop/pcd/trab_final/data/nfl_train.parquet")
df_val   = pd.read_parquet("/Users/llbm/Desktop/pcd/trab_final/data/nfl_test.parquet")
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

X_train, y_train, X_val, y_val, scaler, le, feature_cols = prepare_data(df_train, df_val)
n_features  = X_train.shape[1]
y_flip_cols = get_y_col_indices(feature_cols)
print(f"Input dim: {n_features} | flip Y: {len(y_flip_cols)}")

Rodando no dispositivo: mps
Train: 28086 | Val: 2921
Input dim: 344 | flip Y: 88


In [14]:
BATCH_SIZE = 64
EPOCHS = 50
K = 4    
LR_BODY = 1e-5  
LR_HEAD = 1e-3
DROPOUT = 0.2
SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)
train_dataset = PlayDataset(X_train, y_train, y_flip_cols=y_flip_cols, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           generator=torch.Generator().manual_seed(SEED))
val_dataset = PlayDataset(X_val, y_val, y_flip_cols=y_flip_cols, is_train=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

#loading body
#instanciate MDN (complete)
model = MLPWithMDN(input_dim=n_features, K=K, dropout=DROPOUT).to(device)
#loads trained weights into it
state_orig = torch.load("/Users/llbm/Desktop/pcd/trab_final/models/nfl_mlp_base_full.pt", map_location=device)
# Maps only the body weights (the 3 intermediate layers)
# The original model has net.0, net.1, ..., net.11 (4 blocks × 3 layers)
# The MDN body has body.net.0 ... body.net.11 same structure without the last layer
body_state = {}
for k, v in state_orig.items():
    #skips last linear (net.12 = Linear(256, 199))
    if k.startswith('net.12'):
        continue
    #net.X → body.net.X
    new_key = k.replace('net.', 'body.net.')
    body_state[new_key] = v

missing, unexpected = model.load_state_dict(body_state, strict=False)
optimizer = torch.optim.Adam([
    {'params': model.body.parameters(), 'lr': LR_BODY},
    {'params': model.head.parameters(), 'lr': LR_HEAD},], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = MDNCRPSLoss().to(device)

In [15]:
print(f"Train MDN K={K} componentes | {EPOCHS} epochs")
print(f"LR body: {LR_BODY} | LR head: {LR_HEAD}")

best_val = float('inf')
best_state = None
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_crps, val_cdfs, _ = evaluate_crps(model, val_loader, device)
    scheduler.step()
    lr_body = optimizer.param_groups[0]['lr']
    lr_head = optimizer.param_groups[1]['lr']

    if val_crps < best_val:
        best_val   = val_crps
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"epoch {epoch:02d}/{EPOCHS} | LR body: {lr_body:.1e} | LR head: {lr_head:.1e} | "
              f"Train CRPS: {train_loss:.5f} | Val CRPS: {val_crps:.5f}")

print(f"\n>>Best Val CRPS: {best_val:.5f}")
print(f">> Baseline MLP:    0.01252")

#looking specifically at the tails
model.load_state_dict(best_state)
_, best_cdfs, _ = evaluate_crps(model, val_loader, device)
print_tail_diagnostics(best_cdfs, y_val, label="MDN K=4")

torch.save(best_state, "nfl_mdn_model.pt")
artifacts = {
    'n_features' : n_features,
    'K'          : K,
    'dropout'    : DROPOUT,
    'val_crps'   : best_val,
    'feature_cols': feature_cols,
    'y_flip_cols': y_flip_cols,}
with open("nfl_mdn_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

Train MDN K=4 componentes | 50 epochs
LR body: 1e-05 | LR head: 0.001
epoch 01/50 | LR body: 1.0e-05 | LR head: 1.0e-03 | Train CRPS: 0.01361 | Val CRPS: 0.01258
epoch 10/50 | LR body: 9.1e-06 | LR head: 9.0e-04 | Train CRPS: 0.01281 | Val CRPS: 0.01251
epoch 20/50 | LR body: 6.9e-06 | LR head: 6.5e-04 | Train CRPS: 0.01279 | Val CRPS: 0.01250
epoch 30/50 | LR body: 4.1e-06 | LR head: 3.5e-04 | Train CRPS: 0.01278 | Val CRPS: 0.01251
epoch 40/50 | LR body: 1.9e-06 | LR head: 9.6e-05 | Train CRPS: 0.01276 | Val CRPS: 0.01251
epoch 50/50 | LR body: 1.0e-06 | LR head: 1.0e-06 | Train CRPS: 0.01276 | Val CRPS: 0.01251

>>Best Val CRPS: 0.01249
>> Baseline MLP:    0.01252

CRPS by region (MDN K=4)
Negative plays (< 0):  n= 318 | CRPS=0.01653
Short plays (0 - 5):   n=1874 | CRPS=0.00565
Medium plays (6 - 15): n= 611 | CRPS=0.01685
Long plays (> 15):     n= 118 | CRPS=0.08767

Right-tail calibration
   Threshold | Predicted | Observed |    Error
  yards > 10   |   0.0900 |   0.0880 |  +0.0021

To correct the residual bias in the distribution tails which persisted even in the MDN due to the scarcity of long-play observations, we applied Beta calibration as a post-processing step. The underlying rationale is: while the MDN learns the overall distribution effectively, it systematically underestimates probabilities in the tails because the loss gradient is dominated by observations near the center of the distribution, which are substantially more frequent. Rather than attempting to correct this behavior through retraining (which had already proven unstable in some of our toher experiments using weighted losses and focal loss) we directly adjusted the output probabilities.

"Beta calibration learns a monotonic transformation for each CDF bin using logistic regression, providing sufficient flexibility to correct systematic bias while remaining simple enough to avoid overfitting in regions with limited tail data. This makes it particularly well suited for calibration problems where the underlying predictive model is already strong but exhibits consistent probability distortions."

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
import pickle
import math

In [17]:
class MLPBody(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LayerNorm(512), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(512, 512), nn.LayerNorm(512), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.ELU(), nn.Dropout(dropout),)

    def forward(self, x):
        return self.net(x)

In [18]:
class MDNHead(nn.Module):
    def __init__(self, hidden_dim=256, K=4):
        super().__init__()
        self.K = K
        self.net = nn.Linear(hidden_dim, 3 * K)

    def forward(self, h):
        out = self.net(h)
        pi = F.softmax(out[:, :self.K], dim=-1)
        mu = out[:, self.K:2*self.K]
        sigma = torch.exp(out[:, 2*self.K:]).clamp(min=0.1, max=50.0)
        return pi, mu, sigma


In [19]:
class MLPWithMDN(nn.Module):
    def __init__(self, input_dim, K=4, dropout=0.2):
        super().__init__()
        self.body = MLPBody(input_dim, dropout)
        self.head = MDNHead(hidden_dim=256, K=K)

    def forward(self, x):
        return self.head(self.body(x))

In [20]:
def mixture_cdf(pi, mu, sigma, y_bins):
    pi_e = pi.unsqueeze(2)
    mu_e = mu.unsqueeze(2)
    sigma_e = sigma.unsqueeze(2)
    y_e = y_bins.view(1, 1, -1)
    cdf = (pi_e * gaussian_cdf(y_e, mu_e, sigma_e)).sum(dim=1)
    return torch.clamp(cdf, 0.0, 1.0)

In [21]:
def prepare_data(df_train, df_val):
    drop_cols = ['PlayId', 'GameId', 'Yards']
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    pos_cols = [c for c in feature_cols if 'Position' in c]
    le = LabelEncoder()
    le.fit(pd.concat([df_train[c] for c in pos_cols]).fillna('UNK'))

    df_train = df_train.copy()
    df_val = df_val.copy()
    for col in pos_cols:
        df_train[col] = le.transform(df_train[col].fillna('UNK'))
        df_val[col] = df_val[col].fillna('UNK').apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else 0)

    df_train = df_train.fillna(0.0)
    df_val = df_val.fillna(0.0)
    X_val = df_val[feature_cols].values.astype(np.float32)
    y_val = df_val['Yards'].values.astype(np.float32)
    scaler = StandardScaler()
    scaler.fit(df_train[feature_cols].values.astype(np.float32))
    X_val  = scaler.transform(X_val)

    return X_val, y_val, feature_cols

In [22]:
def get_mdn_cdfs(model, X, device, batch_size=256):
    model.eval()
    y_bins = torch.arange(-99, 100, dtype=torch.float32).to(device)
    all_cdfs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            pi, mu, sigma = model(batch)
            cdfs = mixture_cdf(pi, mu, sigma, y_bins).cpu().numpy()
            all_cdfs.append(cdfs)
    return np.vstack(all_cdfs)

In [23]:
def beta_calibrate_bin(p_pred, y_bin, C=1.0):
    eps = 1e-7
    p_clip = np.clip(p_pred, eps, 1 - eps)
    X_cal = np.column_stack([np.log(p_clip), np.log(1 - p_clip)])
    clf = LogisticRegression(fit_intercept=True, max_iter=200, solver='lbfgs', C=C)
    if y_bin.sum() > 0 and (1 - y_bin).sum() > 0:
        clf.fit(X_cal, y_bin)
        return clf
    return None

def apply_calibrator(clf, p_pred):
    if clf is None:
        return p_pred
    eps = 1e-7
    p_clip = np.clip(p_pred, eps, 1 - eps)
    X_cal = np.column_stack([np.log(p_clip), np.log(1 - p_clip)])
    return clf.predict_proba(X_cal)[:, 1]


def calibrate_cdfs(cdfs_pred, calibrators):
    N, B = cdfs_pred.shape
    cdfs_cal = np.zeros_like(cdfs_pred)
    for b in range(B):
        cdfs_cal[:, b] = apply_calibrator(calibrators[b], cdfs_pred[:, b])
    cdfs_cal = np.maximum.accumulate(cdfs_cal, axis=1)
    return np.clip(cdfs_cal, 0.0, 1.0)

def train_calibrators(cdfs_pred, y_val, y_bins, C=1.0):
    calibrators = []
    for b in range(len(y_bins)):
        y_bin = (y_val <= y_bins[b]).astype(int)
        calibrators.append(beta_calibrate_bin(cdfs_pred[:, b], y_bin, C=C))
    return calibrators

In [24]:
def crps_numpy(cdfs, y_true):
    y_bins   = np.arange(-99, 100, dtype=np.float32)
    cdf_true = (y_bins[None, :] >= y_true[:, None]).astype(np.float32)
    return float(np.mean((cdfs - cdf_true) ** 2))

In [25]:
def print_tail_diagnostics(cdfs, y_val, label=""):
    y_bins_np = np.arange(-99, 100, dtype=np.float32)

    def crps_mask(cdfs, y_true, mask):
        if mask.sum() == 0:
            return float('nan')
        true_cdf = (y_bins_np[None, :] >= y_true[mask, None]).astype(np.float32)
        return float(np.mean((cdfs[mask] - true_cdf) ** 2))

    negative_mask = y_val < 0
    short_mask = (y_val >= 0) & (y_val <= 5)
    medium_mask = (y_val > 5)  & (y_val <= 15)
    long_mask = y_val > 15

    print(f"\nCRPS by region ({label})")
    print(f"Negative plays (< 0):  n={negative_mask.sum():4d} | CRPS={crps_mask(cdfs, y_val, negative_mask):.5f}")
    print(f"Short plays (0 - 5): n={short_mask.sum():4d} | CRPS={crps_mask(cdfs, y_val, short_mask):.5f}")
    print(f"Medium plays (6 - 15): n={medium_mask.sum():4d} | CRPS={crps_mask(cdfs, y_val, medium_mask):.5f}")
    print(f"Long plays (> 15): n={long_mask.sum():4d} | CRPS={crps_mask(cdfs, y_val, long_mask):.5f}")
    print(f"\nRight-tail")
    print(f"  {'Threshold':>10} | {'P(pred)':>8} | {'P(actual)':>8} | {'Error':>8}")

    for threshold in [10, 15, 20, 25, 30, 40]:
        idx = threshold + 99
        pred_prob = float(np.mean(1 - cdfs[:, idx]))
        actual_prob = float(np.mean(y_val > threshold))
        error = pred_prob - actual_prob
        print(f"  yards > {threshold:2d}   | "
            f"{pred_prob:8.4f} | "
            f"{actual_prob:8.4f} | "
            f"{error:+8.4f}")

In [26]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
X_val, y_val, feature_cols = prepare_data(df_train, df_val)
n_features = X_val.shape[1]
print(f"Validation plays: {len(y_val)} | Features: {n_features}")

#Load MDN model
model = MLPWithMDN(input_dim=n_features, K=4, dropout=0.2).to(device)
state = torch.load("nfl_mdn_model.pt", map_location=device)
model.load_state_dict(state)
print("MDN model loaded: nfl_mdn_model.pt")
# Generate MDN CDFs
print("\nGenerating MDN CDFs...")

cdfs_mdn = get_mdn_cdfs(model, X_val, device)
crps_mdn = crps_numpy(cdfs_mdn, y_val)
print(f"MDN CRPS before calibration: {crps_mdn:.5f}")

print_tail_diagnostics(cdfs_mdn, y_val,label="MDN without calibration")

Device: mps
Validation plays: 2921 | Features: 344
MDN model loaded: nfl_mdn_model.pt

Generating MDN CDFs...
MDN CRPS before calibration: 0.01249

CRPS by region (MDN without calibration)
Negative plays (< 0):  n= 318 | CRPS=0.01653
Short plays (0 - 5):   n=1874 | CRPS=0.00565
Medium plays (6 - 15): n= 611 | CRPS=0.01685
Long plays (> 15):     n= 118 | CRPS=0.08767

Right-tail
   Threshold |  P(pred) | P(actual) |    Error
  yards > 10   |   0.0900 |   0.0880 |  +0.0021
  yards > 15   |   0.0286 |   0.0404 |  -0.0118
  yards > 20   |   0.0085 |   0.0195 |  -0.0110
  yards > 25   |   0.0046 |   0.0103 |  -0.0057
  yards > 30   |   0.0036 |   0.0072 |  -0.0036
  yards > 40   |   0.0025 |   0.0045 |  -0.0019


In [27]:
print("BETA CALIBRATION 6-Fold CV on MDN")

y_bins = np.arange(-99, 100, dtype=np.float32)
N_FOLDS = 6
C = 1.0  #same C value from the previous experiment that worked well
kf = KFold(n_splits=N_FOLDS,shuffle=True, random_state=76) #bolduc
cdfs_oof = np.zeros_like(cdfs_mdn)
for fold_idx, (cal_idx, eval_idx) in enumerate(kf.split(X_val)):
    calibrators = train_calibrators(cdfs_mdn[cal_idx], y_val[cal_idx], y_bins, C=C)
    cdfs_oof[eval_idx] = calibrate_cdfs(cdfs_mdn[eval_idx], calibrators)
    crps_before = crps_numpy(cdfs_mdn[eval_idx], y_val[eval_idx])
    crps_after = crps_numpy(cdfs_oof[eval_idx], y_val[eval_idx])
    print(f"Fold {fold_idx + 1}/{N_FOLDS} | "
        f"before: {crps_before:.5f} | "
        f"after: {crps_after:.5f} | "
        f"improvement: {crps_before - crps_after:+.5f}")
crps_calibrated = crps_numpy(cdfs_oof, y_val)

BETA CALIBRATION 6-Fold CV on MDN
Fold 1/6 | before: 0.01310 | after: 0.01310 | improvement: -0.00000
Fold 2/6 | before: 0.01279 | after: 0.01279 | improvement: +0.00000
Fold 3/6 | before: 0.01357 | after: 0.01356 | improvement: +0.00002
Fold 4/6 | before: 0.01081 | after: 0.01080 | improvement: +0.00001
Fold 5/6 | before: 0.01229 | after: 0.01226 | improvement: +0.00003
Fold 6/6 | before: 0.01239 | after: 0.01237 | improvement: +0.00002


In [28]:
print(f"MDN CRPS without calibration: {crps_mdn:.5f}")
print(f"MDN CRPS + OOF Beta calibration: {crps_calibrated:.5f}")
print(f"Improvement: {crps_mdn - crps_calibrated:+.5f}")
print(f"MLP Baseline: 0.01252")
print(f"MLP + Beta calibration: 0.01248")

print_tail_diagnostics(cdfs_oof,y_val,label="MDN + Beta calibration")
# Save final calibrators (trained on all validation data)
final_calibrators = train_calibrators(cdfs_mdn,y_val,y_bins,C=C)
with open("nfl_mdn_calibrators.pkl", "wb") as f:
    pickle.dump({
        "calibrators": final_calibrators,
        "crps_mdn": crps_mdn,
        "crps_cal": crps_calibrated,
        "C": C,
        "y_bins": y_bins,}, f)

print(f"\n>> FINAL CRPS): {crps_calibrated:.5f}")

MDN CRPS without calibration: 0.01249
MDN CRPS + OOF Beta calibration: 0.01248
Improvement: +0.00001
MLP Baseline: 0.01252
MLP + Beta calibration: 0.01248

CRPS by region (MDN + Beta calibration)
Negative plays (< 0):  n= 318 | CRPS=0.01658
Short plays (0 - 5):   n=1874 | CRPS=0.00573
Medium plays (6 - 15): n= 611 | CRPS=0.01665
Long plays (> 15):     n= 118 | CRPS=0.08701

Right-tail
   Threshold |  P(pred) | P(actual) |    Error
  yards > 10   |   0.0880 |   0.0880 |  -0.0000
  yards > 15   |   0.0402 |   0.0404 |  -0.0002
  yards > 20   |   0.0195 |   0.0195 |  -0.0000
  yards > 25   |   0.0102 |   0.0103 |  -0.0001
  yards > 30   |   0.0070 |   0.0072 |  -0.0002
  yards > 40   |   0.0044 |   0.0045 |  -0.0000

>> FINAL CRPS): 0.01248
